# Parse Transfer Capacities

Preparation of transfer capacities. Currently, we take the old values from winter 2010/2011 and convert them to hourly values to be consistent with later approaches

jab 01.06.2019

script checked by Jonas 23.07.2025: works but need to check in create gdx if still needed

## Packages and options

In [1]:
import pandas as pd
import numpy as np

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
fn_in = "../source_data/ntc.xlsx"
fn_out = "../parsed_data/ntc.csv"

## Load data

We get the static, indicative values from 2010/2011

In [3]:
df_in = pd.read_excel(fn_in, sheet_name="NTC_2017", skiprows=1, index_col="From").stack().reset_index()
df_in.columns = ["from", "to", "ntc"]
df_in.head()

,from,to,ntc
0,IE,GB,759.0
1,GB,IE,974.0
2,GB,FR,1736.0
3,GB,NL,997.0
4,PT,ES,2978.0


## Upsample data

Given values are static. So we assign a random date and upsample them to hourly frequency

In [4]:
start = pd.to_datetime("2015/01/01 00:00")
ende = pd.to_datetime("2019/01/01 00:00")
df_start = df_in.copy()
df_end = df_in.copy()
df_start["date"] = start
df_end["date"] = ende
df_ntc_in = pd.concat([df_start, df_end])
df_ntc_in = df_ntc_in.set_index(["date", "from", "to"])

In [5]:
df_ntc = df_ntc_in.unstack().unstack().resample("H").ffill()
#df_ntc.index = df_ntc.index.tz_localize("utc")
df_ntc = df_ntc.stack().stack().reset_index()
df_ntc.head()

C:\Users\jonas\AppData\Local\Temp\ipykernel_13272\2498839421.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_ntc = df_ntc_in.unstack().unstack().resample("H").ffill()
C:\Users\jonas\AppData\Local\Temp\ipykernel_13272\2498839421.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_ntc = df_ntc.stack().stack().reset_index()
C:\Users\jonas\AppData\Local\Temp\ipykernel_13272\2498839421.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_ntc = df_ntc.stack().stack().reset_index()


,date,from,to,ntc
0,2015-01-01,AT,CH,1200.0
1,2015-01-01,AT,CZ,900.0
2,2015-01-01,AT,DE,5000.0
3,2015-01-01,AT,IT,405.0
4,2015-01-01,BE,FR,1800.0


## Export

In [6]:
df_ntc.to_csv(fn_out, encoding="utf-8", index=False)